In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from surprise import SVD, Dataset, Reader
from sklearn.metrics.pairwise import cosine_similarity
import requests
from dotenv import load_dotenv
import os

In [2]:
df_movies = pd.read_csv('../data/movies.csv')
df_ratings = pd.read_csv('../data/ratings.csv')

# Content Based Filtering

In [3]:
#  genre filtering
def movies_by_genres(genre):
    return df_movies[df_movies["genres"].str.contains(genre, case=False)]["title"]

movies_by_genres("comedy")

0                                Toy Story (1995)
2                         Grumpier Old Men (1995)
3                        Waiting to Exhale (1995)
4              Father of the Bride Part II (1995)
6                                  Sabrina (1995)
                          ...                    
9732                    Gintama: The Movie (2010)
9734                          Silver Spoon (2014)
9737    Black Butler: Book of the Atlantic (2017)
9738                 No Game No Life: Zero (2017)
9741          Andrew Dice Clay: Dice Rules (1991)
Name: title, Length: 3756, dtype: object

In [4]:


# Functions

def get_poster(title):

    clean_title = title.split("(")[0].strip()
    year = title.split("(")[1].replace(")","").strip()

    url = "https://api.themoviedb.org/3/search/movie"
    params = {
        "api_key": TMDB_API_KEY,
        "query": clean_title,
        "year": year
    }
    
    response = requests.get(url, params=params)
    data = response.json()

    if not data["results"]:                       # handling edge cases  if the result is empty 
        return None
    else :
        poster_path = data['results'][0]['poster_path']

        full_url =  "https://image.tmdb.org/t/p/w500" + poster_path
        
        return full_url
        




In [5]:
# Content based filtering using TF-IDF
df_movies['features'] = df_movies['title'] + " " + df_movies['genres'].str.replace("|", " ")

df_movies.head(3)

,movieId,title,genres,features
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995) Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance


In [6]:
def get_movie_description(title):
    try:
        clean_title = title.split("(")[0].strip()
        year = title.split("(")[1].replace(")","").strip()
        
        url = "https://api.themoviedb.org/3/search/movie"
        params = {
            "api_key":TMDB_API_KEY,
            "query":  clean_title,
            "year":  year
        }
        
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        if not data["results"]:
            return ""
            
        return data["results"][0].get('overview', "")
        
    except:
        return ""

print(get_movie_description("Toy Story (1995)"))


In [7]:
avg_ratings = df_ratings.groupby('movieId')['rating'].mean()
print(avg_ratings.head())
print(f"Total movies with ratings: {len(avg_ratings)}")

movieId
1    3.920930
2    3.431818
3    3.259615
4    2.357143
5    3.071429
Name: rating, dtype: float64
Total movies with ratings: 9724


In [8]:
def get_cb_recommendations(genre, n=5, random=True):
    
    genre_movies = movies_by_genres(genre)
    
    if len(genre_movies) == 0:
        return []
    
    if random:
        sample = genre_movies.sample(
            min(10, len(genre_movies))
        ).tolist()
    else:
        sample = genre_movies.head(10).tolist()
    
    features = []
    titles = []
    movie_ids = []
    
    # Genre reference
    genre_reference = " ".join([genre] * 5)
    features.append(genre_reference)
    titles.append("REFERENCE")
    movie_ids.append(None)
    
    # Movie features
    for title in sample:
        desc = get_movie_description(title)
        genre_text = df_movies[df_movies['title'] == title]['genres'].values[0].replace("|", " ")
        
        mid = df_movies[df_movies['title'] == title]['movieId'].values[0]
        
        combined = f"{genre_text} {desc}"
        features.append(combined)
        titles.append(title)
        movie_ids.append(mid)
    
    # TF-IDF
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(features)
    
    # Cosine Similarity
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    # Similarity scores vs reference
    sim_scores = list(enumerate(sim_matrix[0]))
    sim_scores = sim_scores[1:]  # Skip reference
    
    # Multiply similarity × avg rating 🎯
    weighted_scores = []
    for idx, sim in sim_scores:
        mid = movie_ids[idx]
        avg_rating = avg_ratings.get(mid, 3.0)
        weighted = sim * avg_rating
        weighted_scores.append((idx, weighted))
    
    # Sort by weighted score
    weighted_scores = sorted(
        weighted_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Return top n
    recommended = [
        titles[i[0]] 
        for i in weighted_scores[:n]
    ]
    return recommended

# Colloaborative Filtering

In [9]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    df_ratings[['userId', 'movieId', 'rating']], 
    reader
)
trainset = data.build_full_trainset()
svd_model = SVD()
svd_model.fit(trainset)

# Test prediction
prediction = svd_model.predict(uid=1, iid=100)
print(f"Predicted rating: {prediction.est}")

Predicted rating: 3.995105920069783


In [10]:
def get_cf_recommendations(user_id, n=5):
    # Step 1 — watched movies
    watched = df_ratings[df_ratings['userId'] == user_id]['movieId'].tolist()
    
    # Step 2 — unwatched movies
    all_movies = df_movies['movieId'].tolist()
    unwatched = [m for m in all_movies 
                 if m not in watched]
    
    # Step 3 — predict ratings
    predictions = []
    for movie_id in unwatched:
        pred = svd_model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Step 4 — sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Step 5 — get top n titles
    titles = []
    for movie_id, score in predictions[:n]:
        title = df_movies[df_movies['movieId'] == movie_id]['title'].values
        if len(title) > 0:
            titles.append(title[0])
    
    return titles

# Hybrid Recommendation Approach

In [ ]:
def get_hybrid_recommendations(user_id, genre, n=5,alpha=0.5,beta=0.5):

    cb_recs = get_cb_recommendations(genre, n=10)
    cf_recs = get_cf_recommendations(user_id, n=10, genre=genre) 
    
    # Assign scores based on position
    # First item = highest score
    
    cb_scores = {}
    for i, movie in enumerate(cb_recs):
        cb_scores[movie] = len(cb_recs) - i
        # Position 0 → highest score
    
    cf_scores = {}
    for i, movie in enumerate(cf_recs):
        cf_scores[movie] = len(cf_recs) - i
        

    # Get all unique movies from both
    all_movies = set(cb_recs + cf_recs)
    
    # Calculate combined score for each
    final_scores = {}
    for movie in all_movies:
        cb_s = cb_scores.get(movie, 0)
        cf_s = cf_scores.get(movie, 0)
        
        final_scores[movie] = (alpha * cb_s) + (beta * cf_s)
    
    # Sort by score — highest first
    sorted_movies = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    # Return top n titles
    return [movie for movie, score in sorted_movies[:n]]

In [12]:
def get_recommendations(genre, n=5, user_id=None):
    '''
    If user_id provided → Hybrid (Personal Mode)
    If user_id is None  → CB only (Guest Mode)
    '''
    
    if user_id is None:
        # Guest Mode — CB only
        return get_cb_recommendations(genre, n=n)
    else:
        # Personal Mode — Hybrid
        return get_hybrid_recommendations(
            user_id, genre, n=n
        )

In [13]:
# Which one did you call?
print("Guest mode : ",get_recommendations("Drama", n=5, user_id=None))
# OR
print("Personal mode : ",get_recommendations("Drama", n=5, user_id=1))

Guest mode :  ['Dinner Rush (2000)', 'Das Experiment (Experiment, The) (2001)', 'Gone Girl (2014)', 'Robin Hood: Prince of Thieves (1991)', 'The Pacific (2010)']
Personal mode :  ['Postman, The (Postino, Il) (1994)', 'Umberto D. (1952)', 'Taxi Driver (1976)', 'Pay It Forward (2000)', 'Tokyo Story (Tôkyô monogatari) (1953)']


In [14]:
print(get_recommendations(
    user_id=None, 
    genre="Drama", 
    n=5
))

['McFarland, USA (2015)', 'Shock Corridor (1963)', 'Wonder Wheel (2017)', 'Man on the Moon (1999)', 'Imaginarium of Doctor Parnassus, The (2009)']


In [15]:
for user_id in [1, 5, 50]:
    recs = get_hybrid_recommendations(
        user_id=user_id,
        genre="Action",
        n=5
    )
    print(f"\nUser {user_id}:")
    for r in recs:
        print(f"  → {r}")


User 1:
  → Postman, The (Postino, Il) (1994)
  → Liability, The (2012)
  → Tokyo Tribe (2014)
  → Taxi Driver (1976)
  → Tombstone (1993)

User 5:
  → Godfather, The (1972)
  → Mesrine: Public Enemy #1 (L'ennemi public n°1) (2008)
  → Ghost and the Darkness, The (1996)
  → Monty Python and the Holy Grail (1975)
  → Juice (1992)

User 50:
  → Bloodsport 2 (a.k.a. Bloodsport II: The Next Kumite) (1996)
  → Rear Window (1954)
  → Grave of the Fireflies (Hotaru no haka) (1988)
  → New York Cop (Nyû Yôku no koppu) (1993)
  → Streetcar Named Desire, A (1951)
